In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('/content/sample_data/diabetes.csv')

# Display the first 5 rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

# Display basic information about the DataFrame
print("\nDataFrame Info:")
df.info()

# Display descriptive statistics of the DataFrame
print("\nDescriptive Statistics:")
display(df.describe())

First 5 rows of the dataset:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Descriptive Statistics:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


The dataset contains several variables related to patient health, including: `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and `Outcome` (which is the target variable, indicating diabetes).

Before proceeding with calculations, it's important to handle any missing values, which often appear as zeros in this dataset for variables like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, where a zero value is physiologically impossible. We will replace these zeros with `NaN` and then decide on an imputation strategy, such as replacing them with the median.

In [2]:
# Replace 0 values with NaN for relevant columns where 0 is not a valid measurement
columns_with_zeros_as_nan = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in columns_with_zeros_as_nan:
    df[col] = df[col].replace(0, np.nan)

# Check for missing values after replacement
print("Missing values after replacing 0s with NaN:")
display(df.isnull().sum())

# Impute missing values with the median of each column
for col in columns_with_zeros_as_nan:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# Verify that there are no more missing values
print("\nMissing values after median imputation:")
display(df.isnull().sum())

print("\nFirst 5 rows after cleaning:")
display(df.head())

Missing values after replacing 0s with NaN:


,0
Pregnancies,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0



Missing values after median imputation:


/tmp/ipykernel_688/415266663.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)


,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0



First 5 rows after cleaning:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


### Point Estimates and Confidence Intervals

Now, let's calculate some point estimates for selected variables. We'll determine the sample mean for `Glucose` and the sample proportion for `Outcome` (diabetes prevalence). Following that, we'll construct 95% confidence intervals for these population parameters.

In [3]:
# 1. Point Estimate: Sample Mean of Glucose
glucose_mean = df['Glucose'].mean()
print(f"Sample Mean of Glucose: {glucose_mean:.2f}")

# 2. Confidence Interval for Population Mean of Glucose (using t-distribution for small sample or unknown population std)
glucose_std = df['Glucose'].std()
glucose_count = df['Glucose'].count()
glucose_sem = glucose_std / np.sqrt(glucose_count) # Standard Error of the Mean

# For a 95% confidence interval, alpha = 0.05, so alpha/2 = 0.025. Degrees of freedom = count - 1
confidence_level = 0.95
alpha = 1 - confidence_level
t_critical = stats.t.ppf(1 - alpha/2, df=glucose_count - 1)

margin_of_error_glucose = t_critical * glucose_sem
confidence_interval_glucose = (glucose_mean - margin_of_error_glucose, glucose_mean + margin_of_error_glucose)
print(f"95% Confidence Interval for Population Mean Glucose: ({confidence_interval_glucose[0]:.2f}, {confidence_interval_glucose[1]:.2f})")

# 3. Point Estimate: Sample Proportion of Diabetes (Outcome == 1)
diabetes_proportion = df['Outcome'].mean() # Since Outcome is 0 or 1, the mean is the proportion
print(f"\nSample Proportion of Diabetes: {diabetes_proportion:.3f}")

# 4. Confidence Interval for Population Proportion of Diabetes
# Using the Wald method for proportion confidence interval
# Standard Error of Proportion = sqrt(p * (1 - p) / n)
n = len(df)
std_err_proportion = np.sqrt(diabetes_proportion * (1 - diabetes_proportion) / n)

# For a 95% confidence interval, z_critical for a two-tailed test is 1.96
z_critical = stats.norm.ppf(1 - alpha/2) # For large samples, normal approximation is often used

margin_of_error_proportion = z_critical * std_err_proportion
confidence_interval_proportion = (diabetes_proportion - margin_of_error_proportion, diabetes_proportion + margin_of_error_proportion)
print(f"95% Confidence Interval for Population Proportion of Diabetes: ({confidence_interval_proportion[0]:.3f}, {confidence_interval_proportion[1]:.3f})")

Sample Mean of Glucose: 121.66
95% Confidence Interval for Population Mean Glucose: (119.50, 123.81)

Sample Proportion of Diabetes: 0.349
95% Confidence Interval for Population Proportion of Diabetes: (0.315, 0.383)


### Hypothesis Testing: Two-Sample t-test

**Research Question:** Is there a significant difference in the mean glucose levels between individuals with diabetes and those without diabetes?

**Hypotheses:**
*   **Null Hypothesis (H0):** There is no significant difference in the mean glucose levels between diabetic and non-diabetic individuals (μ_diabetic = μ_non-diabetic).
*   **Alternative Hypothesis (H1):** There is a significant difference in the mean glucose levels between diabetic and non-diabetic individuals (μ_diabetic ≠ μ_non-diabetic).

**Significance Level (α):** 0.05

In [4]:
# Separate data into two groups based on 'Outcome'
diabetic_glucose = df[df['Outcome'] == 1]['Glucose']
non_diabetic_glucose = df[df['Outcome'] == 0]['Glucose']

# Perform independent two-sample t-test
# We assume unequal variances (Welch's t-test) as a safer default, though an F-test could check for equality
t_statistic, p_value = stats.ttest_ind(diabetic_glucose, non_diabetic_glucose, equal_var=False)

print(f"Two-Sample t-test for Glucose Levels:\n")
print(f"  Mean Glucose (Diabetic): {diabetic_glucose.mean():.2f}")
print(f"  Mean Glucose (Non-Diabetic): {non_diabetic_glucose.mean():.2f}")
print(f"  t-statistic: {t_statistic:.2f}")
print(f"  p-value: {p_value:.3f}")

# Compare p-value with significance level
significance_level = 0.05
print(f"\nSignificance Level (α): {significance_level}")

if p_value < significance_level:
    print("Decision: Reject the null hypothesis.")
    print("Interpretation: There is a statistically significant difference in mean glucose levels between individuals with diabetes and those without diabetes.")
elif p_value > significance_level:
    print("Decision: Fail to reject the null hypothesis.")
    print("Interpretation: There is no statistically significant difference in mean glucose levels between individuals with diabetes and those without diabetes.")

Two-Sample t-test for Glucose Levels:

  Mean Glucose (Diabetic): 142.13
  Mean Glucose (Non-Diabetic): 110.68
  t-statistic: 14.85
  p-value: 0.000

Significance Level (α): 0.05
Decision: Reject the null hypothesis.
Interpretation: There is a statistically significant difference in mean glucose levels between individuals with diabetes and those without diabetes.


### Hypothesis Testing: Chi-square Test

To apply the Chi-square test, we first need to create categorical variables. Let's create `AgeGroup` from the `Age` column.

**Research Question:** Is there a significant association between `AgeGroup` and `Outcome` (diabetes status)?

**Hypotheses:**
*   **Null Hypothesis (H0):** There is no association between `AgeGroup` and `Outcome` (they are independent).
*   **Alternative Hypothesis (H1):** There is an association between `AgeGroup` and `Outcome` (they are dependent).

**Significance Level (α):** 0.05

In [5]:
# Create Age Groups
bins = [20, 30, 40, 50, 60, 70, 80, 90]
labels = ['20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# Create a contingency table
contingency_table = pd.crosstab(df['AgeGroup'], df['Outcome'])
print("Contingency Table (AgeGroup vs. Outcome):\n")
display(contingency_table)

# Perform Chi-square test of independence
chi2_statistic, p_value_chi2, dof, expected_freq = stats.chi2_contingency(contingency_table)

print(f"\nChi-square Test of Independence (AgeGroup vs. Outcome):\n")
print(f"  Chi-square statistic: {chi2_statistic:.2f}")
print(f"  p-value: {p_value_chi2:.3f}")
print(f"  Degrees of freedom: {dof}")

# Compare p-value with significance level
significance_level = 0.05
print(f"\nSignificance Level (α): {significance_level}")

if p_value_chi2 < significance_level:
    print("Decision: Reject the null hypothesis.")
    print("Interpretation: There is a statistically significant association between age group and diabetes status.")
elif p_value_chi2 > significance_level:
    print("Decision: Fail to reject the null hypothesis.")
    print("Interpretation: There is no statistically significant association between age group and diabetes status.")

Contingency Table (AgeGroup vs. Outcome):



Outcome,0,1
AgeGroup,,
20-29,312,84
30-39,89,76
40-49,53,65
50-59,23,34
60-69,21,8
70-79,1,1
80-89,1,0



Chi-square Test of Independence (AgeGroup vs. Outcome):

  Chi-square statistic: 79.65
  p-value: 0.000
  Degrees of freedom: 6

Significance Level (α): 0.05
Decision: Reject the null hypothesis.
Interpretation: There is a statistically significant association between age group and diabetes status.
